# Replication completion timing bounds

This notebook prepares replication-timing profiles, fits initiation-rate landscapes, runs stochastic replication simulations, and compares empirical completion statistics with the analytical bounds of Alkhaled et al. (2026).

1. **Analysis A:** UCSC/ENCODE chromosome profiles treated as non-periodic lines.
2. **Analysis B:** HCT116 chromosome 8 windows treated as synthetic periodic domains.

Data acquisition and timing-profile construction are implemented in `replication_data_sources.py`; fitting, simulation, bounds, and plotting are implemented in `replication_src.py`.


## Imports

Load the numerical and plotting packages together with the two project modules.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from replication_src import *
from replication_data_sources import *

# Analysis A: non-periodic chromosome profiles

UCSC chromosome profiles are treated as observed, non-periodic finite lines. Initiation-rate fitting and simulation use `perQ=False`; the theoretical calculation uses line geometry with non-wrapping intervals over the observed chromosome.


## A1. Load UCSC Repli-seq signal

Select the ENCODE/UW cell lines and chromosomes, download any missing UCSC WaveSignal BigWigs, load them as 1 kb arrays, and cache raw timing CSVs. WaveSignal is already a single wavelet-smoothed replication-timing profile, so no S-phase reconstruction is required. The centromere intervals configured here are applied only when an analysis is run.


In [ ]:
UCSC_REPLISEQ_BASE_URL = "http://hgdownload.soe.ucsc.edu/goldenPath/hg19/encodeDCC/wgEncodeUwRepliSeq"
REPLISEQ_REPLICATE = 1
REPLISEQ_CHROMS = ["chr1"]
REPLISEQ_RESOLUTION = 1_000
REPLISEQ_OVERWRITE_BIGWIG = False
REPLISEQ_OVERWRITE_TIMING_CACHE = False

UCSC_REPLISEQ_CELL_LINE_NAMES = {
    "BG02ES": "Bg02es",
    "BJ": "Bj",
    "HELAS3": "Helas3",
    "HEPG2": "Hepg2",
    "HUVEC": "Huvec",
    "IMR90": "Imr90",
    "K562": "K562",
    "MCF7": "Mcf7",
    "NHEK": "Nhek",
    "SKNSH": "Sknsh",
}
REPLISEQ_CELL_LINES = list(UCSC_REPLISEQ_CELL_LINE_NAMES)
REPLISEQ_CENTROMERES_BP = [(115_000_000, 145_000_000)]


In [ ]:
REPLISEQ_BIGWIGS = download_ucsc_repliseq_wavelet_bigwigs(
    cell_lines=REPLISEQ_CELL_LINES,
    base_url=UCSC_REPLISEQ_BASE_URL,
    replicate=REPLISEQ_REPLICATE,
    data_dir=DATA_DIR,
    overwrite=REPLISEQ_OVERWRITE_BIGWIG,
    cell_line_names=UCSC_REPLISEQ_CELL_LINE_NAMES,
)

display(ucsc_repliseq_bigwig_status_table(
    REPLISEQ_BIGWIGS,
    replicate=REPLISEQ_REPLICATE,
))

In [ ]:
LINE_PROFILE_DATASETS, REPLISEQ_WAVELET_ARRAYS, line_timing_summary = build_ucsc_repliseq_line_datasets(
    cell_lines=REPLISEQ_CELL_LINES,
    chroms=REPLISEQ_CHROMS,
    resolution=REPLISEQ_RESOLUTION,
    replicate=REPLISEQ_REPLICATE,
    data_dir=DATA_DIR,
    timing_dir=TIMING_DIR,
    overwrite_timing_cache=REPLISEQ_OVERWRITE_TIMING_CACHE,
    cell_line_names=UCSC_REPLISEQ_CELL_LINE_NAMES,
)
LINE_KEY = next(iter(LINE_PROFILE_DATASETS))
repliseq_wavelet_signal = REPLISEQ_WAVELET_ARRAYS[LINE_KEY]["signal"]
repliseq_wavelet_positions = REPLISEQ_WAVELET_ARRAYS[LINE_KEY]["positions"]
repliseq_wavelet_bigwig = REPLISEQ_WAVELET_ARRAYS[LINE_KEY]["bigwig"]

display(line_timing_summary)
display(timing_cache_table(LINE_PROFILE_DATASETS))

## A2. Run one full-chromosome analysis

Select one cached UCSC profile and run the non-periodic fit, simulation, and line-bound calculation. Missing values are interpolated before resampling and smoothing. With the current descending `timing_range=(550, 30)`, centromere bins are set before rescaling to the processed profile value that maps to 30 min, then excluded from both empirical and theoretical completion and expected-time calculations.

The source grid is 1 kb. `refine_factor=1` preserves it, while `refine_factor=.1` produces a 10 kb simulation grid. Supply fork speed in kb/min; `run_single_dataset` converts it to grid sites/min.


In [ ]:
LINE_ANALYSIS_CELL_LINE = "SKNSH"
LINE_ANALYSIS_CHROM = "chr1"

LINE_ANALYSIS_CELL_LINE_KEY = str(LINE_ANALYSIS_CELL_LINE).upper().replace("-", "").replace("_", "").replace(" ", "")
LINE_KEY = f"{LINE_ANALYSIS_CELL_LINE_KEY}_line_{safe_filename(LINE_ANALYSIS_CHROM)}"

if LINE_KEY not in LINE_PROFILE_DATASETS:
    available = ", ".join(LINE_PROFILE_DATASETS)
    raise KeyError(f"{LINE_KEY} is not available. Available keys: {available}")

line_selected_cfg = LINE_PROFILE_DATASETS[LINE_KEY]
display(pd.DataFrame([{
    "key": LINE_KEY,
    "cell_line": line_selected_cfg["cell_line"],
    "chromosome": line_selected_cfg["chrom"],
    "resolution_bp": line_selected_cfg["resolution"],
    "source_bigwig": line_selected_cfg["source_bigwig"],
}]))

LINE_ANALYSIS_SIM_NUMBER = 1000
LINE_ANALYSIS_REFINE_FACTOR = .1
LINE_ANALYSIS_SMOOTH_WINDOW = 100
LINE_ANALYSIS_TIMING_RANGE = (550, 30)
LINE_ANALYSIS_EPS_GRID = np.geomspace(1e-2, 0.99, 100)
LINE_ANALYSIS_NUM_T_BOUND = 4_000
LINE_ANALYSIS_MAX_REP_TIME = 2_000

line_result = run_single_dataset(
    line_selected_cfg,
    fork_speed_kb_min=1.4,
    sim_number=LINE_ANALYSIS_SIM_NUMBER,
    refine_factor=LINE_ANALYSIS_REFINE_FACTOR,
    smooth_window=LINE_ANALYSIS_SMOOTH_WINDOW,
    timing_range=LINE_ANALYSIS_TIMING_RANGE,
    eps_grid=LINE_ANALYSIS_EPS_GRID,
    num_t_bound=LINE_ANALYSIS_NUM_T_BOUND,
    max_rep_time=LINE_ANALYSIS_MAX_REP_TIME,
    timing_cache_mode="load",
    centromeres_bp=REPLISEQ_CENTROMERES_BP,
)

## A3. Plot and summarize the selected result

Plot the fitted initiation rate, prepared Repli-seq timing versus simulated mean timing, and theoretical versus empirical $T_\varepsilon$. Then report the empirical and theoretical expected-time summary.


In [ ]:
make_standard_plots(
    line_result,
    save_figures=False,
    initiation_ylims=(1e-4, 1e-2),

)

expected_time_summary_table({LINE_KEY: line_result})

## A4. Compare UCSC cell lines on chromosome 1

Run the selected chr1 cell lines with the A2 preparation settings and compare the empirical $\max_x \mathbb{E}[T(x)]$ with the line-geometry expected-time bound. The same centromere mask is excluded from both quantities.


In [ ]:
LINE_BATCH_CELL_LINES = REPLISEQ_CELL_LINES

LINE_BATCH_CHROM = "chr1"
LINE_BATCH_SIM_NUMBER = 1000
LINE_BATCH_REFINE_FACTOR = globals().get("LINE_ANALYSIS_REFINE_FACTOR", .1)
LINE_BATCH_SMOOTH_WINDOW = globals().get("LINE_ANALYSIS_SMOOTH_WINDOW", 100)
LINE_BATCH_TIMING_RANGE = globals().get("LINE_ANALYSIS_TIMING_RANGE", (550, 30))
LINE_BATCH_EPS_GRID = globals().get("LINE_ANALYSIS_EPS_GRID", np.geomspace(1e-2, 0.99, 100))
LINE_BATCH_NUM_T_BOUND = globals().get("LINE_ANALYSIS_NUM_T_BOUND", 4_000)
LINE_BATCH_MAX_REP_TIME = globals().get("LINE_ANALYSIS_MAX_REP_TIME", 2_000)

LINE_BATCH_KEYS = [
    f"{str(cell_line).upper().replace('-', '').replace('_', '').replace(' ', '')}_line_{safe_filename(LINE_BATCH_CHROM)}"
    for cell_line in LINE_BATCH_CELL_LINES
]
missing_line_batch_keys = [key for key in LINE_BATCH_KEYS if key not in LINE_PROFILE_DATASETS]
if missing_line_batch_keys:
    available = ", ".join(LINE_PROFILE_DATASETS)
    missing = ", ".join(missing_line_batch_keys)
    raise KeyError(f"Missing batch dataset key(s): {missing}. Available keys: {available}")

LINE_BATCH_DATASETS = {key: LINE_PROFILE_DATASETS[key] for key in LINE_BATCH_KEYS}
display(pd.DataFrame([
    {
        "key": key,
        "cell_line": cfg["cell_line"],
        "chromosome": cfg["chrom"],
        "resolution_bp": cfg["resolution"],
        "source_bigwig": cfg["source_bigwig"],
    }
    for key, cfg in LINE_BATCH_DATASETS.items()
]))

line_batch_results = run_dataset_collection(
    LINE_BATCH_DATASETS,
    selected_keys=LINE_BATCH_KEYS,
    fork_speed_kb_min=1.4,
    sim_number=LINE_BATCH_SIM_NUMBER,
    refine_factor=LINE_BATCH_REFINE_FACTOR,
    smooth_window=LINE_BATCH_SMOOTH_WINDOW,
    timing_range=LINE_BATCH_TIMING_RANGE,
    eps_grid=LINE_BATCH_EPS_GRID,
    num_t_bound=LINE_BATCH_NUM_T_BOUND,
    max_rep_time=LINE_BATCH_MAX_REP_TIME,
    timing_cache_mode="load",
    centromeres_bp=REPLISEQ_CENTROMERES_BP,
)

In [ ]:
fig, ax, line_expected_time_pairs = plot_expected_time_pair_scatter(
    line_batch_results,
    title=f"Full line {LINE_BATCH_CHROM}: expected local replication-time bound across cell lines",
    label_col="cell_line",
    xlims=(0, 800),
    ylims=(0, 800),
)
fig.savefig(FIGURE_DIR / f"line_expected_time_pairs_{safe_filename(LINE_BATCH_CHROM)}.pdf", bbox_inches="tight")
plt.show()

# Analysis B: periodic HCT116 chromosome 8 windows

The three chromosome 8 windows defined below use HCT116 Repli-seq data from Zhao et al. (GSE137764) and regions motivated by Jaworski et al. Each window is analyzed as a synthetic circular domain for comparison with the torus bound, not as a purified ecDNA Repli-seq measurement.


## B1. Prepare Zhao HCT116 timing profiles

Download the processed 50 kb HCT116 matrix, calculate a weighted S1-S16 phase index in each source bin, and linearly interpolate each selected window onto a 1 kb analysis grid. The interpolation provides the simulation grid but does not increase the measurement resolution. Cache one timing CSV per window.


In [ ]:
ZHAO_HCT116_GSE_ACCESSION = DEFAULT_ZHAO_HCT116_GSE_ACCESSION
ZHAO_HCT116_REPLISEQ_URL = DEFAULT_ZHAO_HCT116_REPLISEQ_URL
ZHAO_HCT116_REPLISEQ_GZ = DATA_DIR / DEFAULT_ZHAO_HCT116_REPLISEQ_FILENAME_GZ
ZHAO_HCT116_REPLISEQ_TABLE = DATA_DIR / DEFAULT_ZHAO_HCT116_REPLISEQ_FILENAME
ZHAO_HCT116_SOURCE_RESOLUTION = DEFAULT_ZHAO_HCT116_SOURCE_RESOLUTION
ZHAO_HCT116_ANALYSIS_RESOLUTION = DEFAULT_ZHAO_HCT116_ANALYSIS_RESOLUTION
ZHAO_HCT116_S_PHASE_BINS = DEFAULT_ZHAO_HCT116_S_PHASE_BINS
ZHAO_HCT116_OVERWRITE_DOWNLOAD = False
ZHAO_HCT116_OVERWRITE_TIMING_CACHE = False

zhao_hct116_table_path = ensure_zhao_hct116_repliseq(
    url=ZHAO_HCT116_REPLISEQ_URL,
    gz_path=ZHAO_HCT116_REPLISEQ_GZ,
    table_path=ZHAO_HCT116_REPLISEQ_TABLE,
    overwrite=ZHAO_HCT116_OVERWRITE_DOWNLOAD,
)
zhao_hct116_matrix = load_zhao_hct116_repliseq_matrix(
    path=zhao_hct116_table_path,
    s_phase_bins=ZHAO_HCT116_S_PHASE_BINS,
    source_resolution=ZHAO_HCT116_SOURCE_RESOLUTION,
)

PERIODIC_CELL_LINES = ["HCT116"]
PERIODIC_REGIONS = [
    {
        "region_id": "CHR8_CMYC_7MB",
        "label": "chr8 c-MYC/ecDNA context window",
        "short_label": "chr8 c-MYC/ecDNA 7 Mb",
        "chrom": "chr8",
        "start": 124_000_000,
        "end": 130_775_000,
    },
    {
        "region_id": "CHR8_ECDNA_DOMINANT",
        "label": "chr8 dominant ecDNA-derived interval",
        "short_label": "chr8 dominant ecDNA interval",
        "chrom": "chr8",
        "start": 126_425_747,
        "end": 128_512_820,
    },
    {
        "region_id": "CHR8_CMYC_ORIGIN_ZOOM",
        "label": "chr8 c-MYC-centred origin-density zoom",
        "short_label": "chr8 c-MYC origin zoom",
        "chrom": "chr8",
        "start": 127_675_747,
        "end": 127_875_747,
    },
]

PERIODIC_INTERVAL_DATASETS = build_zhao_hct116_periodic_configs(
    PERIODIC_REGIONS,
    resolution=ZHAO_HCT116_ANALYSIS_RESOLUTION,
    accession=ZHAO_HCT116_GSE_ACCESSION,
    source_resolution=ZHAO_HCT116_SOURCE_RESOLUTION,
    cell_line="HCT116",
)

display(pd.DataFrame([
    {
        "key": key,
        "cell_line": cfg["cell_line"],
        "region": cfg["short_label"].replace(f"{cfg['cell_line']} ", ""),
        "coordinates": f"{cfg.get('requested_chrom', cfg['chrom'])}:{cfg['start']}-{cfg['end']}",
        "source_resolution_bp": cfg["source_resolution"],
        "analysis_resolution_bp": cfg["resolution"],
    }
    for key, cfg in PERIODIC_INTERVAL_DATASETS.items()
]))

display(timing_cache_table(PERIODIC_INTERVAL_DATASETS))

periodic_timing_preprocessing = preprocess_zhao_hct116_timing_collection(
    PERIODIC_INTERVAL_DATASETS,
    zhao_hct116_matrix,
    overwrite=ZHAO_HCT116_OVERWRITE_TIMING_CACHE,
)
display(periodic_timing_preprocessing)

## B2. Run one periodic-window analysis

Select one cached window and run periodic smoothing, initiation-rate fitting, and simulation with `perQ=True`, followed by the torus-bound calculation. `refine_factor=1` preserves the cached 1 kb analysis grid.


In [ ]:
PERIODIC_EXAMPLE_CELL_LINE = "HCT116"
PERIODIC_EXAMPLE_REGION_ID = "CHR8_CMYC_ORIGIN_ZOOM"
PERIODIC_KEY = f"{PERIODIC_EXAMPLE_CELL_LINE}_{PERIODIC_EXAMPLE_REGION_ID}"

if PERIODIC_KEY not in PERIODIC_INTERVAL_DATASETS:
    available = ", ".join(PERIODIC_INTERVAL_DATASETS)
    raise KeyError(f"{PERIODIC_KEY} is not available. Available keys: {available}")

periodic_result = run_single_dataset(
    PERIODIC_INTERVAL_DATASETS[PERIODIC_KEY],
    fork_speed_kb_min=1.4,
    sim_number=100000,
    refine_factor=1,
    smooth_window=10,
    timing_range=(60, 10),
    eps_grid=np.geomspace(1e-2, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

## B3. Plot and summarize the selected result

Plot the periodic initiation rate, prepared HCT116 timing versus simulated mean timing, and torus bound versus empirical $T_\varepsilon$. Then report the expected-time summary.


In [ ]:
make_standard_plots(
    periodic_result,
    save_figures=False,
)

expected_time_summary_table({PERIODIC_KEY: periodic_result})

## B4. Compare the three periodic windows

Run all three HCT116 windows with the batch settings below and compare empirical $\max_x \mathbb{E}[T(x)]$ with the torus expected-time bound.


In [ ]:
PERIODIC_BATCH_DATASETS = PERIODIC_INTERVAL_DATASETS

periodic_batch_results = run_dataset_collection(
    PERIODIC_BATCH_DATASETS,
    selected_keys=list(PERIODIC_BATCH_DATASETS),
    fork_speed_kb_min=1.4,
    sim_number=1000,
    refine_factor=1,
    smooth_window=50,
    timing_range=(60, 10),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

In [ ]:
fig, ax, periodic_expected_time_pairs = plot_expected_time_pair_scatter(
    periodic_batch_results,
    title="Torus: expected local replication-time bound for HCT116 chr8 finite windows",
    xlims=(0, 100),
    ylims=(0, 100),
)
fig.savefig(FIGURE_DIR / "hct116_chr8_three_windows_torus_expected_time_pairs.pdf", bbox_inches="tight")
plt.show()

# Notes on interpretation

Analysis A uses a finite, non-wrapping implementation of the line geometry over the observed chromosome. Its configured centromere bins are assigned the earliest rescaled timing value and omitted from both empirical and theoretical completion and expected-time calculations.

Analysis B circularizes each selected HCT116 genomic window. Periodic smoothing, fitting, simulation, and torus local-mass calculations therefore wrap across the window boundary. These profiles are derived from genomic HCT116 Repli-seq data and should not be interpreted as direct ecDNA measurements.

Both workflows convert physical fork speed to grid units using

$$
v_{\mathrm{grid}} = \frac{v_{\mathrm{kb/min}}}{dx_{\mathrm{kb}}}.
$$

The expected-time bound is obtained by integrating the completion-time survival bound and is compared with the empirical slowest local mean, $\max_x \mathbb{E}[T(x)]$. Initiation-rate fitting follows [Berkemeier et al. (2025)](https://www.nature.com/articles/s41467-025-59991-w); full references for the data and completion bounds are listed in the README.
